# Stability selection and the candidate sets (Prompt #9, Steps 16-19, 41-46, 51-59)

50 resamples of half the development quarters plus odd/even years and early/middle/late thirds. A run keeps the features that enter before the first noise probe (at most 20); a feature is selection-stable at frequency >= 0.6. Minimal / Standard / Extended are prefixes of one ranking (stable features first); their sizes come from the validation plateau (0.90 / 0.97 of the best rank IC, reversion targets excluded).

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.selection` and `xauusd_quant.research.feature_selection_*` and read back from `results/feature_selection/`. Generate those first (needs the Prompt #8 research of the same timeframe for the pipeline-null veto):

```
xq feature-select --timeframe 5m                        # every stage + report + manifests
xq feature-select --timeframe 5m --target residual_reduction
xq feature-select --timeframe 5m --target future_return --horizon 5
xq feature-selection-report --timeframe 5m              # summary, SEL-H ledger rows, plots
xq build-feature-manifest --set standard --timeframe 5m # verify + reproduce the matrix
```

**Periods.** Development 2003-2017 selects; validation 2018-2021 only evaluates what development chose and reads the plateaus; the reserved test period (2022-) is never loaded with outcomes. Diagnostic models are linear (ridge, Lasso, elastic net, L1 logistic) on training-CDF rank designs - not the ML system of Prompt #10, and nothing here is a trading rule.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.selection.config import load_selection_config

scfg = load_selection_config(ROOT / "config" / "feature_selection.yaml")

TIMEFRAME = "5m"       # 5m | 15m (configured); 30m | 1h if they were run

out = scfg.results_path / TIMEFRAME
if not out.exists():
    print(f"No selection for {TIMEFRAME}. Run:  xq feature-select --timeframe {TIMEFRAME}")

def table(name):
    """Load a previously generated selection table (None if it was not written)."""
    path = out / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = out / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no plot {path.name})")

def load_json(name):
    path = out / name
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
sets = load_json("sets.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Selection frequency

In [ ]:
plot('selection_frequency')

## How many features enter before a probe

In [ ]:
pl.DataFrame(summary.get('probe_stopping', []))

## Year subsets vs the full development period

In [ ]:
table('year_subset_stability')

## Feature count vs validation rank IC, plateau

In [ ]:
plot('feature_count_curve')
sets.get('plateau')

## The sets

In [ ]:
{k: (len(v), sets['justified'][k]) for k, v in sets.get('sets', {}).items()}

In [ ]:
table('candidate_sets').filter(pl.col('set') == 'standard')

## Recent relevance, regime robustness, target specificity (Steps 42-44)

In [ ]:
display(table('recent_relevance').group_by('recent_relevance').len())
display(table('regime_robustness'))
table('target_specificity')

## Selected features' yearly IC (after selection; reserved years never loaded)

In [ ]:
plot('selected_yearly_ic')

## Manifests

In [ ]:
for p in sorted((out / 'manifests').glob('*.json')):
    m = json.loads(p.read_text())
    print(m['feature_set_id'], m['feature_count'], 'features, max history', m['max_history_required'],
          'effective rank', round(m['effective_rank'] or 0, 2), m['content_hash'])